# Drex Max v1 (27B) — Out-of-Distribution Calibration Run (Nov 1–8, 2026 arXiv)

Reproducible Colab GPU notebook (`NVIDIA A100-SXM4-80GB` High-RAM, `79.25 GiB` VRAM, `bfloat16`; also supports `G4`/`H100`) for **Step 3** and **Step 4** of [`DavidFlorez/decision-confidence`](https://github.com/DavidFlorez/decision-confidence).

- **Model**: [`nace-ai/drex-max-v1`](https://huggingface.co/nace-ai/drex-max-v1) (`27B` non-generative decision head over `Qwen/Qwen3.8-27B` with non-causal full attention)
- **Dataset**: `data/papers.jsonl` — 560 unseen arXiv CS preprints submitted Nov 1–8, 2026 (96 `fit` + 464 `test` across 8 categories)
- **Outputs**: `results/drex_smoke_test.json`, `results/drex_max.jsonl`, `results/drex_max_reversed.jsonl`, `results/drex_run_meta.json`


In [ ]:
# 1. Verify GPU & CUDA runtime
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'CUDA GPU required for Drex Max v1 (27B) bf16'
print('Device:', torch.cuda.get_device_name(0), '| VRAM (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


In [ ]:
# 2. Clone repository (if running standalone on Colab)
import os, subprocess
if not os.path.exists('/content/decision-confidence') and not os.path.exists('data/papers.jsonl'):
    subprocess.run(['git', 'clone', 'https://github.com/DavidFlorez/decision-confidence.git', '/content/decision-confidence'], check=True)
    os.chdir('/content/decision-confidence')
print('Working directory:', os.getcwd())


In [ ]:
# 3. Download nace-ai/drex-max-v1 snapshot and record commit SHA + wall-clock download time
import json, time
from pathlib import Path
from huggingface_hub import HfApi, snapshot_download

weights_dir = Path('weights/drex-max-v1')
weights_dir.mkdir(parents=True, exist_ok=True)
info = HfApi().model_info('nace-ai/drex-max-v1')
t0 = time.perf_counter()
snapshot_download(repo_id='nace-ai/drex-max-v1', local_dir=str(weights_dir))
dl_sec = time.perf_counter() - t0
(weights_dir / 'hf_download_info.json').write_text(
    json.dumps({'commit_sha': info.sha, 'download_wall_sec': dl_sec}, indent=2) + '\n'
)
print(f'Downloaded nace-ai/drex-max-v1 @ {info.sha} in {dl_sec:.1f}s')


In [ ]:
# 4. Install model requirements (transformers + flash-linear-attention + accelerate)
!pip install -q -r weights/drex-max-v1/requirements.txt


In [ ]:
# 5. Run CLI reference check (H100 bf16 tolerance <= 0.03) + 50-paper dataset smoke test (< 1.0 s/paper)
!python3 scripts/run_drex.py --weights weights/drex-max-v1 --papers data/papers.jsonl --results-dir results --batch-size 8 --mode smoke


In [ ]:
# 6. Run full 560-paper canonical pass + 560-paper reversed option-order pass
!python3 scripts/run_drex.py --weights weights/drex-max-v1 --papers data/papers.jsonl --results-dir results --batch-size 8 --mode full --skip-cli-smoke


In [ ]:
# 7. Verify output counts & summary metadata
import json
from pathlib import Path

for fname in ['drex_max.jsonl', 'drex_max_reversed.jsonl']:
    p = Path('results') / fname
    n = sum(1 for line in p.read_text().splitlines() if line.strip())
    print(f'{fname}: {n} rows ({p.stat().st_size:,} bytes)')

meta = json.loads(Path('results/drex_run_meta.json').read_text())
print('Full run:', json.dumps(meta['full_run'], indent=2))
print('Reversed run:', json.dumps(meta['reversed_run'], indent=2))
